In [1]:
import matplotlib.pyplot as plt
import numpy as np

%matplotlib widget

import Comparison

In [2]:
def OpeningAngle(true_x, true_y, true_z, reco_x, reco_y, reco_z):

    reco_dir = np.column_stack((reco_x, reco_y, reco_z))
    true_dir = np.column_stack((true_x, true_y, true_z))

    dot_product = np.sum(reco_dir * true_dir, axis=1)

    reco_mag = np.linalg.norm(reco_dir, axis=1)
    true_mag = np.linalg.norm(true_dir, axis=1)

    # Avoid division by zero
    valid = (reco_mag > 0) & (true_mag > 0)

    opening_angle = np.full(len(reco_mag), np.nan)

    cos_theta = dot_product[valid] / (reco_mag[valid] * true_mag[valid])

    # Avoid numerical precision errors
    cos_theta = np.clip(cos_theta, -1.0, 1.0)

    opening_angle[valid] = np.degrees(np.arccos(cos_theta))

    return opening_angle

In [3]:
import uproot

file_name = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/files/atmo_ccnusel.root'
with uproot.open(file_name) as file:
    tree = file['ccnuselection/ccnusel']
    nusel_branches = tree.arrays(['Run', 'SubRun', 'Event', 'BeamPdg', 'NuPdg', 'NC', 'TargetZ', 
                                  'RecoNuVtxX', 'RecoNuVtxY', 'RecoNuVtxZ',
                                  'RecoPFPTruePDG', 'RecoPFPTrackShowerScore', 'RecoPFPRecoNHits', 'RecoPFPTruePrimary',
                                  'RecoTrackRecoStartDirX', 'RecoTrackRecoStartDirY', 'RecoTrackRecoStartDirZ',
                                  'RecoPFPTrueDirX', 'RecoPFPTrueDirY', 'RecoPFPTrueDirZ',
                                  'RecoShowerRecoDirX', 'RecoShowerRecoDirY', 'RecoShowerRecoDirZ'], library='ak')

In [7]:
import awkward as ak
# Match shape to PFP jagged array
mc_pdg = nusel_branches['RecoPFPTruePDG']
mc_iscc = ak.broadcast_arrays(nusel_branches['NC'], mc_pdg)[0]
mc_nupdg = ak.broadcast_arrays(nusel_branches['NuPdg'], mc_pdg)[0]
mc_isprimary = nusel_branches['RecoPFPTruePrimary']
mc_hastruedir = (nusel_branches['RecoPFPTrueDirX'] > -1.5) & (nusel_branches['RecoPFPTrueDirY'] > -1.5) & (nusel_branches['RecoPFPTrueDirZ'] > -1.5)
mc_hasrecodir = (nusel_branches['RecoTrackRecoStartDirX'] > -1.5) & (nusel_branches['RecoTrackRecoStartDirY'] > -1.5) & (nusel_branches['RecoTrackRecoStartDirZ'] > -1.5)

target_mask = (mc_iscc == 0) & (abs(mc_nupdg) == 14) & (abs(mc_pdg) == 13) & (mc_isprimary == 1) & (mc_hastruedir) & (mc_hasrecodir)

In [8]:

pfp_true_x = ak.to_numpy(ak.flatten(nusel_branches['RecoPFPTrueDirX'][target_mask]))
pfp_true_y = ak.to_numpy(ak.flatten(nusel_branches['RecoPFPTrueDirY'][target_mask]))
pfp_true_z = ak.to_numpy(ak.flatten(nusel_branches['RecoPFPTrueDirZ'][target_mask]))


# pfp_reco_x = ak.to_numpy(ak.flatten(nusel_branches['RecoShowerRecoDirX'][target_mask]))
# pfp_reco_y = ak.to_numpy(ak.flatten(nusel_branches['RecoShowerRecoDirY'][target_mask]))
# pfp_reco_z = ak.to_numpy(ak.flatten(nusel_branches['RecoShowerRecoDirZ'][target_mask]))

pfp_reco_x = ak.to_numpy(ak.flatten(nusel_branches['RecoTrackRecoStartDirX'][target_mask]))
pfp_reco_y = ak.to_numpy(ak.flatten(nusel_branches['RecoTrackRecoStartDirY'][target_mask]))
pfp_reco_z = ak.to_numpy(ak.flatten(nusel_branches['RecoTrackRecoStartDirZ'][target_mask]))

opening_angle = OpeningAngle(pfp_true_x, pfp_true_y, pfp_true_z, pfp_reco_x, pfp_reco_y, pfp_reco_z)

In [9]:
n_entries = opening_angle.shape[0]
hist_counts, bin_edges = np.histogram(opening_angle, bins=18, range=[0,180])
hist_fraction = hist_counts / n_entries

# Plot with error == sqrt(n_i)/N
hist_error = np.sqrt(hist_counts) / n_entries
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])  

fig, ax = plt.subplots()
ax.step(bin_centers, hist_fraction, where='mid', color='blue', linewidth=1, label=f'electron')
ax.fill_between(bin_centers, hist_fraction, step='mid',color='blue', alpha=0.3)
ax.errorbar(bin_centers, hist_fraction, yerr=hist_error, fmt='none', ecolor='blue', capsize=2)

ax.set_title('CCNumu - Primary - Muon')
ax.set_xlabel('True-Reco Direction Opening Angle [degrees]')
ax.set_ylabel('Frac. of Particles')
ax.grid(True)

fig.savefig(f'/Users/isobel/Desktop/AtmoPlots/Atmo_AllEnergies_DirOpeningAngle_Muon.png', format='png', bbox_inches='tight', dpi=300)
plt.close(fig)   

data = {'hist_counts': hist_counts, 'hist_fraction': hist_fraction, 'hist_error': hist_error, 'bin_centers': bin_centers}
np.savez('/Users/isobel/Desktop/AtmoPlots/Atmo_AllEnergies_DirOpeningAngle_Muon.npz', data)


In [ ]:
int = 'CCNue'
tier = 'Primary'
particle = 'Electron'
plot_type = 'Completeness_Purity'

#file = f'PFPValidation/MC_OnlyReco/Completeness/{int}_{tier}_{particle}.npz'
file = f'PFPValidation/2D/{plot_type}/{int}_{tier}_{particle}.npz'

file_name_OG_grow = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_OGGrow_OGMatch/{file}'
file_name_DL_grow = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_DLGrow_OGMatch/{file}'

data_OG_grow = dict(np.load(file_name_OG_grow))
data_DL_grow = dict(np.load(file_name_DL_grow))

if (int == 'AllInt') :
    int='CCNumu+CCNue+NC'

plot_title = f'{int} - {tier} - {particle}'
file_name = f'LBL_AllEnergies_AfterShowerGrowing_{particle}_{plot_type}'
# Comparison.PlotDistOverlay(data_2019, 'MicroBooNE era configuration at DUNE (2019)', 
#                            data_2026_extra, 'RS&DC stage 2 configuration', plot_title, 'upper center', file_name)
# Comparison.PlotDistOverlay(data_OG_grow, 'MicroBooNE era shower growing', 
#                            data_DL_grow, 'Transformer based shower growing', plot_title, 'upper center', file_name)


Comparison.Plot2DHist(data_DL_grow, f'Transformer based shower growing: \n {plot_title}', file_name)


In [ ]:
# int = 'NC'

# file = f'EventValidation/Reco/NuVertexAccuracy/{int}.npz'
# plot_title = f'{int}'

# file_name_10kt = f'/Users/isobel/Desktop/DUNE/2026/Dom3/Plots_10kt/{file}'
# file_name_1x8x6 = f'/Users/isobel/Desktop/DUNE/2026/Dom3/Plots_1x8x6/{file}'

# loaded_10kt = np.load(file_name_10kt)
# data_10kt = dict(loaded_10kt)

# loaded_1x8x6 = np.load(file_name_1x8x6)
# data_1x8x6 = dict(loaded_1x8x6)

# Comparison.PlotDistOverlay(data_1x8x6, '1x8x6', data_10kt, '10kt', plot_title)

In [ ]:
# int = 'NC'

# file = f'EventValidation/Diff/VertexdR_All/{int}.npz'
# plot_title = f'{int}'

# file_name_10kt = f'/Users/isobel/Desktop/DUNE/2026/Dom3/Plots_10kt/{file}'
# file_name_1x8x6 = f'/Users/isobel/Desktop/DUNE/2026/Dom3/Plots_1x8x6/{file}'

# loaded_10kt = np.load(file_name_10kt)
# data_10kt = dict(loaded_10kt)

# loaded_1x8x6 = np.load(file_name_1x8x6)
# data_1x8x6 = dict(loaded_1x8x6)

# Comparison.PlotDistOverlay(data_1x8x6, '1x8x6', data_10kt, '10kt', plot_title)

In [ ]:
int = 'AllInt'
tier = 'Primary'
particle = 'Photon'
plot_type = 'Completeness'

#file = f'PFPValidation/MC_OnlyReco/Completeness/{int}_{tier}_{particle}.npz'
file = f'PFPValidation/MC/{plot_type}/{int}_{tier}_{particle}.npz'
#file = f'PFPValidation/Diff/DeltaLength/{int}_{tier}_{particle}.npz'
#file = f'PFPValidation/BM/VertexAcc/{int}_{tier}_{particle}.npz'

# file_name_2019 = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2018_Sub1GeVVis/{file}'
# file_name_2026_extra = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2026_extra_Sub1GeVVis/{file}'
# file_name_2026_extra = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2026_extra_balanced/{file}'
# file_name_2019 = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2018_Sub1GeVVis/{file}'
# file_name_2026 = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2026_Sub1GeVVis/{file}'
# file_name_2026_extra = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2026_extra_Sub1GeVVis/{file}'

file_name_atmo_2026_extra = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_atmo_2026_Sub1GeVVis/{file}'

# file_name_OG_grow = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_OGGrow_OGMatch/{file}'
# file_name_DL_grow = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_DLGrow_OGMatch/{file}'

# loaded_2019 = np.load(file_name_2019)
# data_2019 = dict(loaded_2019)

# loaded_2026 = np.load(file_name_2026)
# data_2026 = dict(loaded_2026)

# loaded_2026_extra = np.load(file_name_2026_extra)
# data_2026_extra = dict(loaded_2026_extra)

data_OG_grow = dict(np.load(file_name_OG_grow))
data_DL_grow = dict(np.load(file_name_DL_grow))

if (int == 'AllInt') :
    int='CCNumu+CCNue+NC'

plot_title = f'{int} - {tier} - {particle}'
file_name = f'LBL_AllEnergies_ShowerGrowing_{particle}_{plot_type}'
# Comparison.PlotDistOverlay(data_2019, 'MicroBooNE era configuration at DUNE (2019)', 
#                            data_2026_extra, 'RS&DC stage 2 configuration', plot_title, 'upper center', file_name)
Comparison.PlotDistOverlay(data_OG_grow, 'MicroBooNE era shower growing', 
                           data_DL_grow, 'Transformer based shower growing', plot_title, 'upper center', file_name)

In [ ]:
# int = 'AllInt'
# tier = 'Primary'
# particle = 'Muon'

# file = f'TrackValidation/Track/BM/EndpointAcc/{int}_{tier}_{particle}.npz'
# plot_title = f'{int} - {tier} - {particle}'

# file_name_10kt = f'/Users/isobel/Desktop/DUNE/2026/Dom3/Plots_10kt/{file}'
# file_name_1x8x6 = f'/Users/isobel/Desktop/DUNE/2026/Dom3/Plots_1x8x6/{file}'
# #file_name_1x8x6 = f'/Users/isobel/Desktop/DUNE/2026/Dom3/Plots_10kt_not_cathode/{file}'

# loaded_10kt = np.load(file_name_10kt)
# data_10kt = dict(loaded_10kt)

# loaded_1x8x6 = np.load(file_name_1x8x6)
# data_1x8x6 = dict(loaded_1x8x6)

# #print(data_1x8x6)

# Comparison.PlotDistOverlay(data_1x8x6, '1x8x6', data_10kt, '10kt', plot_title)

In [ ]:
# int = 'CCNue'
# tier = 'Primary'
# particle = 'Electron'

# file = f'ShowerValidation/BM/InitialDirAccuracy/{int}_{tier}_{particle}.npz'
# plot_title = f'{int} - {tier} - {particle}'

# file_name_10kt = f'/Users/isobel/Desktop/DUNE/2026/Dom3/Plots_10kt/{file}'
# file_name_1x8x6 = f'/Users/isobel/Desktop/DUNE/2026/Dom3/Plots_1x8x6/{file}'
# #file_name_1x8x6 = f'/Users/isobel/Desktop/DUNE/2026/Dom3/Plots_10kt_not_cathode/{file}'

# loaded_10kt = np.load(file_name_10kt)
# data_10kt = dict(loaded_10kt)

# loaded_1x8x6 = np.load(file_name_1x8x6)
# data_1x8x6 = dict(loaded_1x8x6)

# #print(data_1x8x6)

# Comparison.PlotDistOverlay(data_1x8x6, '1x8x6', data_10kt, '10kt', plot_title, 'upper right')

In [ ]:
int = 'CCNue'
tier = 'Primary'
particle = 'Electron'
plot_type = 'TrueVisEnergy'

#file = f'PFPValidation/Efficiency/TrueEnergy/{int}_{tier}_{particle}.npz'
file = f'PFPValidation/Efficiency/{plot_type}/{int}_{tier}_{particle}.npz'

# file_name_2019 = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2018_balanced/{file}'
# file_name_2026 = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2026_balanced/{file}'
# file_name_2026_extra = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2026_extra_balanced/{file}'
#file_name_2019 = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2018/{file}'
# file_name_2026 = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2026/{file}'
#file_name_2026_extra = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2026_extra/{file}'
# file_name_2019 = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2018_Sub1GeVVis/{file}'
# file_name_2026 = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2026_Sub1GeVVis/{file}'
# file_name_2026_extra = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2026_extra_Sub1GeVVis/{file}'

file_name_OG_grow = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_OGGrow_OGMatch/{file}'
file_name_DL_grow = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_DLGrow_OGMatch/{file}'

# loaded_2019 = np.load(file_name_2019)
# data_2019 = dict(loaded_2019)

# loaded_2026 = np.load(file_name_2026)
# data_2026 = dict(loaded_2026)

# loaded_2026_extra = np.load(file_name_2026_extra)
# data_2026_extra = dict(loaded_2026_extra)

data_OG_grow = dict(np.load(file_name_OG_grow))
data_DL_grow = dict(np.load(file_name_DL_grow))

if (int == 'AllInt') :
    int='CCNumu+CCNue+NC'

plot_title = f'{int} - {tier} - {particle}' # Change all int to be numu/nue/nc
file_name = f'LBL_AllEnergies_ShowerGrowing_{particle}_Efficiency_{plot_type}'


# Comparison.PlotEfficiencyOverlay(data_2019, 'MicroBooNE era configuration at DUNE (2019)', 
#                               data_2026_extra, 'RS&DC stage 2 configuration', plot_title, 'upper right', file_name)

Comparison.PlotEfficiencyOverlay(data_OG_grow, 'MicroBooNE era shower growing', 
                           data_DL_grow, 'Transformer based shower growing', plot_title, 'upper right', file_name)

In [ ]:
int = 'CCNue'
tier = 'Primary'
particle = 'Electron'
plot_type_X = 'VisEnergy'
plot_type_Y = 'Purity'

file = f'PFPValidation/XProfile/{plot_type_Y}_{plot_type_X}/{int}_{tier}_{particle}.npz'

# file_name_2018 = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2018/{file}'
# file_name_2026_extra = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2026_extra/{file}'
# file_name_2018 = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2018_balanced/{file}'
# file_name_2026 = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2026_balanced/{file}'
# file_name_2026_extra = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_2026_extra_balanced/{file}'
file_name_OG_grow = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_OGGrow_OGMatch/{file}'
file_name_DL_grow = f'/Users/isobel/Desktop/DUNE/2026/LArPython/LArMetrics/Plots_DLGrow_OGMatch/{file}'

# loaded_2018 = np.load(file_name_2018)
# data_2018 = dict(loaded_2018)

# loaded_2026 = np.load(file_name_2026)
# data_2026 = dict(loaded_2026)

# loaded_2026_extra = np.load(file_name_2026_extra)
# data_2026_extra = dict(loaded_2026_extra)

data_OG_grow = dict(np.load(file_name_OG_grow))
data_DL_grow = dict(np.load(file_name_DL_grow))

if (int == 'AllInt') :
    int='CCNumu+CCNue+NC'
    
plot_title = f'{int} - {tier} - {particle}'
file_name = f'LBL_AllEnergies_ShowerGrowing_{particle}_{plot_type_Y}_{plot_type_X}'
# Comparison.PlotProfileOverlay(data_2018, 'MicroBooNE era configuration at DUNE (2019)', 
#                               data_2026_extra, 'RS&DC stage 2 configuration', plot_title, 'upper right', file_name)

Comparison.PlotProfileOverlay(data_OG_grow, 'MicroBooNE era shower growing', 
                           data_DL_grow, 'Transformer based shower growing', plot_title, 'upper right', file_name)